# Experimentations sur tous les algorithmes

## Chargement de la dataset

In [1]:
from datasets.load import load_msrcv1

name_dataset = "msrcv1"
dataset = load_msrcv1("datasets/")
k = 7
dataset.keys()

dict_keys(['X', 'Y', 'name'])

In [2]:
from bagging import bagging_prime
from metriques import clusteringMeasure

## Fonction utile à l'expérimentation

In [3]:
from numpy import mean, std


def experiment_complet(dataset, k, models, nbexec=10):
    results = []
    for name_model in models.keys():
        perfs = {"ACC": [], "NMI": [], "PUR": []}
        for _ in range(nbexec):
            r = models[name_model](dataset["X"], k)
            perf = clusteringMeasure(dataset["Y"], r)
            for key in perfs.keys():
                perfs[key].append(perf[key])
        
        r = {}
        for key in perfs.keys():
            r[key] = mean(perfs[key])
            r[key + "_std"] = std(perfs[key])

        r["model"] = name_model
        results.append(r)

    return results

## Construction des modèles de façon adéquate pour les expérimentations

In [6]:
from utils.mvgl.scratch.mvgl import mvgl
from utils.mcles.scratch.mcles import mcles


def mcles_adapt(X, k):
    r = mcles(X, k)
    return r["labels"]


def mvgl_adapt(X, k):
    r = mvgl(X, k)
    return r["labels"]


def bagging_mcles(X, k):
    return bagging_prime(X, k, typeweak="mcles", nbreweak=160, p=0.7, nInitForKmeans=5)


def bagging_mvgl(X, k):
    return bagging_prime(X, k, typeweak="mvgl", nbreweak=160, p=0.55)


models = {
    "mvgl": mvgl_adapt,
    "mcles": mcles_adapt,
    "Bag_mvgl": bagging_mvgl,
    "Bag_mcles": bagging_mcles,
}

## Execution des experimentations

In [7]:
perfs = experiment_complet(dataset, k, models)

## Sauvegarde des resultats

In [8]:
from pandas import DataFrame
def save_experiment(results, path_name):
    df = DataFrame(results).sort_values(by=["ACC", "NMI", "PUR"], ascending=False)
    df.to_csv(path_name)
    return df

In [9]:
df = save_experiment(perfs, "perfs.csv")